In [ ]:
"""
Knowledge Distillation — Script 4: Self-Distillation
=====================================================
Model   : ResNet-50 (modified for CIFAR-10) — same architecture as baseline
Dataset : CIFAR-10
Method  : Self-Distillation — the model teaches itself, no external teacher

WHAT IS SELF-DISTILLATION?
===========================
No external teacher is needed. The model uses its own predictions as a
training signal. Acts as a powerful regulariser — the same architecture
trained with self-KD can outperform its own "teacher" version.

Three variants implemented:

1. Born-Again Networks (BAN) — Furlanello 2018
   Train a sequence of K models (generations). Generation k is distilled
   from the soft predictions of generation k-1:
     L_k = CE(y, p_k) + tau^2 * KL(p_{k-1} || p_k)
   p_{k-1} is the fully-trained previous generation (frozen).
   Final prediction: ensemble average of all K generations.
   We train K=3 generations here (gen0 = fine-tuned baseline, gen1+gen2 = distilled).

2. BYOT — Be Your Own Teacher (Zhang 2019)
   Deep layers of the SAME model teach its shallower layers.
   Adds auxiliary classifiers at layer2, layer3, and the main output.
   The deepest classifier (main output) is the "teacher" for the shallower ones:
     L_BYOT = CE(y, p_main) + Σ_{l<L} [CE(y, p_l) + KL(p_main || p_l)]
   Only the main logits are used at inference — no extra cost.

3. CS-KD — Class-wise Self-Knowledge Distillation (Yun 2020)
   Enforce consistent predictions across two augmented views of the SAME CLASS.
   Given two images from the same class (xa, xb):
     L_CSKD = CE(y, p_a) + tau^2 * KL(p_a.detach() || p_b)
   This is easy to implement by grouping batch samples by label.

Sweeps (per variant):
  BAN  — generations K: [2, 3]  |  temperature tau: [2, 4]
  BYOT — aux weight β: [0.5, 1.0, 2.0]
  CS-KD — temperature tau: [2, 4, 8]

Output : __4__KD_self_distillation.json
"""

import os, sys, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch {torch.__version__}", flush=True)
print(f"[init] Device  : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    print(f"[init] GPU     : {torch.cuda.get_device_name(0)}", flush=True)

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__4__KD_self_distillation.json"

BATCH_SIZE  = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS = 0
PIN_MEMORY  = False
NUM_CLASSES = 10
KD_EPOCHS   = 10
USE_AMP     = (DEVICE.type == "cuda")

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] AMP: {USE_AMP}", flush=True)


# ── Model builder ──────────────────────────────────────────────────────────────
def build_resnet50_cifar(num_classes: int = 10) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path: str) -> nn.Module:
    print(f"[load] Loading baseline from {path} ...", flush=True)
    model = build_resnet50_cifar(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    print("[load] OK", flush=True)
    return model


# ── BYOT model with auxiliary classifiers ──────────────────────────────────────
class BYOTResNet50(nn.Module):
    """
    ResNet-50 with auxiliary classifiers at layer2 and layer3.
    Returns (main_logits, [aux2_logits, aux3_logits]).
    At inference, use only main_logits.
    """
    def __init__(self, base: nn.Module, num_classes: int = 10):
        super().__init__()
        # Copy all layers from the base ResNet-50
        self.conv1   = base.conv1
        self.bn1     = base.bn1
        self.relu    = base.relu
        self.maxpool = base.maxpool
        self.layer1  = base.layer1
        self.layer2  = base.layer2
        self.layer3  = base.layer3
        self.layer4  = base.layer4
        self.avgpool = base.avgpool
        self.fc      = base.fc

        # Auxiliary classifiers (small: global avg pool + linear)
        self.aux2 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, num_classes),   # layer2 output channels in ResNet-50
        )
        self.aux3 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(1024, num_classes),  # layer3 output channels in ResNet-50
        )

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        f2 = x                            # hook point for aux2
        x = self.layer3(x)
        f3 = x                            # hook point for aux3
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        main = self.fc(x)
        aux2 = self.aux2(f2)
        aux3 = self.aux3(f3)
        return main, [aux2, aux3]


# ── Data ────────────────────────────────────────────────────────────────────────
def get_train_loader():
    transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=True,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=512, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# ── Helpers ─────────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def evaluate_standard(model: nn.Module, loader: DataLoader) -> dict:
    """Evaluate a standard ResNet (single output)."""
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        out = model(inputs)
        logits = out[0] if isinstance(out, tuple) else out
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_inference_ms(model: nn.Module, runs: int = 30) -> float:
    m = copy.deepcopy(model).to(DEVICE).eval()
    dummy = torch.randn(128, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(5): m(dummy)
        sync()
        t0 = time.perf_counter()
        for _ in range(runs): m(dummy)
        sync()
    return (time.perf_counter() - t0) / runs * 1000


# ══════════════════════════════════════════════════════════════════════════════
# BAN training loop
# ══════════════════════════════════════════════════════════════════════════════
def run_ban(baseline_model, train_loader, test_loader, baseline_acc,
            num_generations=3, tau=4.0, n_epochs=KD_EPOCHS) -> dict:
    print(f"\n  ┌─ [BAN / K={num_generations} / tau={tau}]", flush=True)

    try:
        generations  = []   # list of trained models
        all_epoch_acc = []

        # Generation 0: start from baseline (already trained)
        prev_gen = copy.deepcopy(baseline_model).to(DEVICE)
        prev_gen.eval()
        print(f"      [gen 0] Using pre-trained baseline as gen-0 teacher", flush=True)
        generations.append(prev_gen)

        t_total_start = time.perf_counter()

        for gen_idx in range(1, num_generations):
            print(f"\n      ── Generation {gen_idx} ──────────────────────────────────", flush=True)
            # Fresh model for this generation (same arch)
            curr_gen = build_resnet50_cifar(NUM_CLASSES).to(DEVICE)

            optimizer = torch.optim.SGD(curr_gen.parameters(), lr=0.01,
                                        momentum=0.9, weight_decay=1e-4, nesterov=True)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
            scaler    = torch.amp.GradScaler(enabled=USE_AMP)

            if DEVICE.type == "cuda":
                torch.cuda.reset_peak_memory_stats()

            epoch_acc = []
            t_gen_start = time.perf_counter()

            for epoch in range(n_epochs):
                curr_gen.train()
                correct = total = 0
                t_ep = time.perf_counter()

                for batch_idx, (inputs, targets) in enumerate(train_loader):
                    inputs  = inputs.to(DEVICE, non_blocking=True)
                    targets = targets.to(DEVICE, non_blocking=True)

                    optimizer.zero_grad(set_to_none=True)

                    with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                        with torch.no_grad():
                            teacher_logits = prev_gen(inputs)
                        student_logits = curr_gen(inputs)

                        ce_loss  = F.cross_entropy(student_logits, targets)
                        p_t      = F.softmax(teacher_logits / tau, dim=1)
                        p_s_log  = F.log_softmax(student_logits / tau, dim=1)
                        kl_loss  = F.kl_div(p_s_log, p_t, reduction="batchmean") * (tau ** 2)
                        loss     = 0.5 * ce_loss + 0.5 * kl_loss

                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()

                    correct += (student_logits.detach().argmax(1) == targets).sum().item()
                    total   += targets.size(0)

                scheduler.step()
                acc = correct / total
                epoch_acc.append(round(acc, 6))
                ep_time = time.perf_counter() - t_ep
                print(f"      [gen {gen_idx}][epoch {epoch+1}/{n_epochs}]  "
                      f"acc={acc:.4f}  time={ep_time:.1f}s", flush=True)

            all_epoch_acc.append(epoch_acc)
            generations.append(curr_gen)
            prev_gen = curr_gen   # this gen becomes teacher for next
            prev_gen.eval()

        sync()
        train_time_s = time.perf_counter() - t_total_start
        peak_gpu_mb  = (round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                        if DEVICE.type == "cuda" else None)

        # Evaluate last generation
        print("      [eval] Last generation ...", flush=True)
        last_metrics = evaluate_standard(generations[-1], test_loader)

        # Evaluate ensemble of all generations
        print("      [eval] Ensemble of all generations ...", flush=True)
        all_logits_list = []
        with torch.no_grad():
            for inputs, _ in test_loader:
                inputs = inputs.to(DEVICE, non_blocking=True)
                batch_logits = []
                for g in generations:
                    g.eval()
                    out = g(inputs)
                    logits = out[0] if isinstance(out, tuple) else out
                    batch_logits.append(logits.cpu())
                # Ensemble: average softmax
                avg = torch.stack([F.softmax(l, dim=1) for l in batch_logits]).mean(0)
                all_logits_list.append(avg.argmax(1).numpy())

        ens_preds  = np.concatenate(all_logits_list)
        ens_labels = np.concatenate([l.numpy() for _, l in test_loader])
        ens_acc    = float(accuracy_score(ens_labels, ens_preds))

        inference_ms = measure_inference_ms(generations[-1])
        size_mb      = model_size_mb(generations[-1])
        acc_drop     = baseline_acc - last_metrics["accuracy"]
        ens_drop     = baseline_acc - ens_acc

        print(f"  └─ LastGen Acc={last_metrics['accuracy']:.4f}  "
              f"Ensemble Acc={ens_acc:.4f}  Drop={acc_drop:+.4f}  "
              f"Train={train_time_s:.1f}s", flush=True)

        return {
            "sweep"              : "BAN",
            "variant"            : "ban",
            "num_generations"    : num_generations,
            "temperature"        : tau,
            "epochs_per_gen"     : n_epochs,
            "train_device"       : str(DEVICE),
            "use_amp"            : USE_AMP,
            "train_time_s"       : round(train_time_s, 2),
            "all_epoch_acc"      : all_epoch_acc,
            "accuracy"           : round(last_metrics["accuracy"], 6),
            "ensemble_accuracy"  : round(ens_acc, 6),
            "precision"          : round(last_metrics["precision"], 6),
            "recall"             : round(last_metrics["recall"], 6),
            "f1"                 : round(last_metrics["f1"], 6),
            "accuracy_drop"      : round(acc_drop, 6),
            "ensemble_drop"      : round(ens_drop, 6),
            "size_mb"            : round(size_mb, 4),
            "inference_ms"       : round(inference_ms, 4),
            "peak_gpu_memory_mb" : peak_gpu_mb,
            "description"        : (
                f"BAN self-distillation: {num_generations} generations, "
                f"tau={tau}, epochs={n_epochs}"
            ),
            "status": "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {"variant": "ban", "num_generations": num_generations,
                "status": "failed", "error": str(e), "accuracy": None, "accuracy_drop": None}


# ══════════════════════════════════════════════════════════════════════════════
# BYOT training loop
# ══════════════════════════════════════════════════════════════════════════════
def run_byot(baseline_model, train_loader, test_loader, baseline_acc,
             beta=1.0, tau=4.0, n_epochs=KD_EPOCHS) -> dict:
    print(f"\n  ┌─ [BYOT / beta={beta} / tau={tau}]", flush=True)

    try:
        base = copy.deepcopy(baseline_model)
        model = BYOTResNet50(base, NUM_CLASSES).to(DEVICE)

        optimizer = torch.optim.SGD(model.parameters(), lr=0.01,
                                    momentum=0.9, weight_decay=1e-4, nesterov=True)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
        scaler    = torch.amp.GradScaler(enabled=USE_AMP)

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        epoch_train_acc = []
        t_start = time.perf_counter()

        for epoch in range(n_epochs):
            model.train()
            correct = total = 0
            t_ep = time.perf_counter()

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    main_logits, aux_logits_list = model(inputs)

                    # Task loss on main output
                    ce_main = F.cross_entropy(main_logits, targets)

                    # BYOT: main (deep) teaches each auxiliary (shallow)
                    kd_loss = torch.tensor(0.0, device=DEVICE)
                    p_main  = F.softmax(main_logits.detach() / tau, dim=1)
                    for aux_logits in aux_logits_list:
                        ce_aux  = F.cross_entropy(aux_logits, targets)
                        p_aux   = F.log_softmax(aux_logits / tau, dim=1)
                        kl      = F.kl_div(p_aux, p_main, reduction="batchmean") * (tau**2)
                        kd_loss = kd_loss + ce_aux + beta * kl

                    loss = ce_main + kd_loss

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                correct += (main_logits.detach().argmax(1) == targets).sum().item()
                total   += targets.size(0)

                if (batch_idx + 1) % 100 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done = batch_idx + 1
                    eta  = elapsed / done * (len(train_loader) - done)
                    print(f"      [epoch {epoch+1}/{n_epochs}] "
                          f"batch {done}/{len(train_loader)}  "
                          f"acc={correct/total:.4f}  ETA={eta:.0f}s", flush=True)

            scheduler.step()
            acc = correct / total
            epoch_train_acc.append(round(acc, 6))
            ep_time = time.perf_counter() - t_ep
            remaining = n_epochs - (epoch + 1)
            print(f"      [epoch {epoch+1}/{n_epochs}] DONE  "
                  f"acc={acc:.4f}  time={ep_time:.1f}s  "
                  f"ETA_remaining={ep_time*remaining:.0f}s", flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start
        peak_gpu_mb  = (round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                        if DEVICE.type == "cuda" else None)

        # Evaluate using only main output (aux classifiers discarded at inference)
        print("      [eval] Evaluating main output only ...", flush=True)
        metrics      = evaluate_standard(model, test_loader)
        inference_ms = measure_inference_ms(model)
        size_mb      = model_size_mb(model)
        acc_drop     = baseline_acc - metrics["accuracy"]

        print(f"  └─ Acc={metrics['accuracy']:.4f}  Drop={acc_drop:+.4f}  "
              f"Train={train_time_s:.1f}s", flush=True)

        return {
            "sweep"             : "BYOT",
            "variant"           : "byot",
            "beta"              : beta,
            "temperature"       : tau,
            "aux_classifiers"   : ["layer2", "layer3"],
            "epochs"            : n_epochs,
            "train_device"      : str(DEVICE),
            "use_amp"           : USE_AMP,
            "train_time_s"      : round(train_time_s, 2),
            "epoch_train_acc"   : epoch_train_acc,
            "accuracy"          : round(metrics["accuracy"],  6),
            "precision"         : round(metrics["precision"], 6),
            "recall"            : round(metrics["recall"],    6),
            "f1"                : round(metrics["f1"],        6),
            "accuracy_drop"     : round(acc_drop, 6),
            "size_mb"           : round(size_mb, 4),
            "inference_ms"      : round(inference_ms, 4),
            "peak_gpu_memory_mb": peak_gpu_mb,
            "description"       : f"BYOT: beta={beta}, tau={tau}, epochs={n_epochs}",
            "status": "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {"variant": "byot", "beta": beta,
                "status": "failed", "error": str(e), "accuracy": None, "accuracy_drop": None}


# ══════════════════════════════════════════════════════════════════════════════
# CS-KD training loop
# ══════════════════════════════════════════════════════════════════════════════
def run_cskd(baseline_model, train_loader, test_loader, baseline_acc,
             tau=4.0, n_epochs=KD_EPOCHS) -> dict:
    """
    CS-KD: within each batch, pair samples of the same class and enforce
    consistent predictions. Uses a batch-level same-class grouping trick.
    """
    print(f"\n  ┌─ [CS-KD / tau={tau}]", flush=True)

    try:
        model = copy.deepcopy(baseline_model).to(DEVICE)

        optimizer = torch.optim.SGD(model.parameters(), lr=0.01,
                                    momentum=0.9, weight_decay=1e-4, nesterov=True)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
        scaler    = torch.amp.GradScaler(enabled=USE_AMP)

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        epoch_train_acc = []
        t_start = time.perf_counter()

        for epoch in range(n_epochs):
            model.train()
            correct = total = 0
            t_ep = time.perf_counter()

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    logits  = model(inputs)
                    ce_loss = F.cross_entropy(logits, targets)

                    # CS-KD: for each class, enforce consistency between pairs
                    # Group indices by class label
                    kd_loss = torch.tensor(0.0, device=DEVICE)
                    num_pairs = 0
                    for cls in targets.unique():
                        idx = (targets == cls).nonzero(as_tuple=True)[0]
                        if idx.numel() < 2:
                            continue
                        # Pair up: even ↔ odd
                        n_pairs = idx.numel() // 2
                        idx_a   = idx[:n_pairs]
                        idx_b   = idx[n_pairs:n_pairs*2]
                        p_a     = F.log_softmax(logits[idx_a] / tau, dim=1)
                        p_b     = F.softmax(logits[idx_b].detach() / tau, dim=1)
                        kd_loss = kd_loss + F.kl_div(p_a, p_b, reduction="batchmean") * (tau**2)
                        num_pairs += n_pairs

                    if num_pairs > 0:
                        kd_loss = kd_loss / num_pairs

                    loss = ce_loss + kd_loss

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                correct += (logits.detach().argmax(1) == targets).sum().item()
                total   += targets.size(0)

                if (batch_idx + 1) % 100 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done = batch_idx + 1
                    eta  = elapsed / done * (len(train_loader) - done)
                    print(f"      [epoch {epoch+1}/{n_epochs}] "
                          f"batch {done}/{len(train_loader)}  "
                          f"acc={correct/total:.4f}  ETA={eta:.0f}s", flush=True)

            scheduler.step()
            acc = correct / total
            epoch_train_acc.append(round(acc, 6))
            ep_time = time.perf_counter() - t_ep
            remaining = n_epochs - (epoch + 1)
            print(f"      [epoch {epoch+1}/{n_epochs}] DONE  "
                  f"acc={acc:.4f}  time={ep_time:.1f}s  "
                  f"ETA_remaining={ep_time*remaining:.0f}s", flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start
        peak_gpu_mb  = (round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                        if DEVICE.type == "cuda" else None)

        print("      [eval] Evaluating ...", flush=True)
        metrics      = evaluate_standard(model, test_loader)
        inference_ms = measure_inference_ms(model)
        size_mb      = model_size_mb(model)
        acc_drop     = baseline_acc - metrics["accuracy"]

        print(f"  └─ Acc={metrics['accuracy']:.4f}  Drop={acc_drop:+.4f}  "
              f"Train={train_time_s:.1f}s", flush=True)

        return {
            "sweep"             : "CS-KD",
            "variant"           : "cskd",
            "temperature"       : tau,
            "epochs"            : n_epochs,
            "train_device"      : str(DEVICE),
            "use_amp"           : USE_AMP,
            "train_time_s"      : round(train_time_s, 2),
            "epoch_train_acc"   : epoch_train_acc,
            "accuracy"          : round(metrics["accuracy"],  6),
            "precision"         : round(metrics["precision"], 6),
            "recall"            : round(metrics["recall"],    6),
            "f1"                : round(metrics["f1"],        6),
            "accuracy_drop"     : round(acc_drop, 6),
            "size_mb"           : round(size_mb, 4),
            "inference_ms"      : round(inference_ms, 4),
            "peak_gpu_memory_mb": peak_gpu_mb,
            "description"       : f"CS-KD: tau={tau}, epochs={n_epochs}",
            "status": "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {"variant": "cskd", "temperature": tau,
                "status": "failed", "error": str(e), "accuracy": None, "accuracy_drop": None}


# ── Main ────────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Self-Distillation (BAN + BYOT + CS-KD)")
    print("  Model: ResNet-50 (modified) — same arch, no external teacher")
    print(f"  Device: {DEVICE}  |  Epochs: {KD_EPOCHS}  |  Batch: {BATCH_SIZE}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline acc: {baseline_acc:.4f}\n", flush=True)

    baseline_model  = load_baseline(BASELINE_MODEL_PATH)
    baseline_size_mb = model_size_mb(baseline_model)
    print(f"  Baseline — Size: {baseline_size_mb:.2f} MB  "
          f"Params: {param_count(baseline_model):,}\n", flush=True)

    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    results = []

    # ── BAN ───────────────────────────────────────────────────────────────────
    print("  ── BAN sweeps ─────────────────────────────────────────────────────", flush=True)
    for K in [2, 3]:
        for tau in [2.0, 4.0]:
            row = run_ban(baseline_model, train_loader, test_loader,
                          baseline_acc, num_generations=K, tau=tau, n_epochs=KD_EPOCHS)
            results.append(row)

    # ── BYOT ──────────────────────────────────────────────────────────────────
    print("\n  ── BYOT sweeps ────────────────────────────────────────────────────", flush=True)
    for beta in [0.5, 1.0, 2.0]:
        row = run_byot(baseline_model, train_loader, test_loader,
                       baseline_acc, beta=beta, tau=4.0, n_epochs=KD_EPOCHS)
        results.append(row)

    # ── CS-KD ─────────────────────────────────────────────────────────────────
    print("\n  ── CS-KD sweeps ───────────────────────────────────────────────────", flush=True)
    for tau in [2.0, 4.0, 8.0]:
        row = run_cskd(baseline_model, train_loader, test_loader,
                       baseline_acc, tau=tau, n_epochs=KD_EPOCHS)
        results.append(row)

    # ── Best overall ──────────────────────────────────────────────────────────
    valid = [r for r in results if r.get("accuracy") is not None]
    if not valid:
        print("  ✗ No valid results.", flush=True)
        return
    best = max(valid, key=lambda r: r["accuracy"])

    report = {
        "method"       : "self_distillation",
        "description"  : "Self-Distillation — BAN / BYOT / CS-KD (no external teacher)",
        "model_arch"   : "ResNet-50 (CIFAR-10 modified)",
        "dataset"      : "CIFAR-10",
        "train_device" : str(DEVICE),
        "use_amp"      : USE_AMP,
        "kd_epochs"    : KD_EPOCHS,
        "baseline"     : baseline,
        "model_info"   : {
            "size_mb" : round(baseline_size_mb, 4),
            "params"  : param_count(baseline_model),
        },
        "best_config"  : {
            "variant"      : best["variant"],
            "accuracy"     : best["accuracy"],
            "accuracy_drop": best["accuracy_drop"],
        },
        "sweeps"       : {
            "BAN"  : "K in [2,3] x tau in [2,4]",
            "BYOT" : "beta in [0.5, 1.0, 2.0], tau=4",
            "CSKD" : "tau in [2, 4, 8]",
        },
        "results"      : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    print(f"  Best: variant={best['variant']} | Acc={best['accuracy']:.4f} | "
          f"Drop={best['accuracy_drop']:+.4f}")


if __name__ == "__main__":
    main()

[init] PyTorch 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] GPU     : NVIDIA GeForce RTX 5070 Laptop GPU
[init] AMP: True

  Self-Distillation (BAN + BYOT + CS-KD)
  Model: ResNet-50 (modified) — same arch, no external teacher
  Device: cuda  |  Epochs: 10  |  Batch: 128

  Baseline acc: 0.9320

[load] Loading baseline from ../__2__baseline_resnet50_cifar10.pth ...
[load] OK
  Baseline — Size: 90.03 MB  Params: 23,520,842

  ── BAN sweeps ─────────────────────────────────────────────────────

  ┌─ [BAN / K=2 / tau=2.0]
      [gen 0] Using pre-trained baseline as gen-0 teacher

      ── Generation 1 ──────────────────────────────────
      [gen 1][epoch 1/10]  acc=0.3003  time=53.7s
      [gen 1][epoch 2/10]  acc=0.4943  time=48.7s
      [gen 1][epoch 3/10]  acc=0.5887  time=48.8s
      [gen 1][epoch 4/10]  acc=0.6543  time=50.0s
      [gen 1][epoch 5/10]  acc=0.6987  time=49.2s
      [gen 1][epoch 6/10]  acc=0.7374  time=49.0s
      [gen 1][epoch 7/10]  acc=0.7672  time=48.8s
